# Cheat Sheet — Aircraft Fair Market Value Prediction (Extended Project)

Quick-reference syntax for every technique used in this project. Snippets use small dummy data so every cell runs standalone — copy the *pattern* into your working notebook.

Sections: Setup · Inspection · The `read_csv` "None" Trap · Unit/Text Cleaning · Ordinal Encoding · Domain Features · EDA & Leakage Checks · Encoding · Models (incl. log-linear) · Tuning & CV · Evaluation · Feature Importance · Deal-Finding Backtest · Persistence.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Loading & Inspection

In [ ]:
df = pd.read_csv('aircraft_valuation.csv')
df.head()
df.shape
df.info()
df.describe().T


## 2. The `read_csv` "None"-as-NaN Trap

In [ ]:
# By default, pandas treats several literal strings as missing values, INCLUDING the word "None":
# '', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN',
# '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null'
#
# If a column has a GENUINE category literally named "None" (e.g. "no damage history"),
# a plain pd.read_csv will silently turn it into NaN.

demo = pd.DataFrame({'damage': ['None', 'Minor', 'Major']})
demo.to_csv('_demo.csv', index=False)
reloaded = pd.read_csv('_demo.csv')
print(reloaded['damage'].tolist())          # ['None' got eaten -> NaN, 'Minor', 'Major'] -- BUG!
print(reloaded['damage'].isnull().sum())    # reports 1 missing value that isn't really missing

# Fix #1: prevent the default NA parsing for that column / entirely
safe = pd.read_csv('_demo.csv', keep_default_na=False, na_values=[''])
print(safe['damage'].tolist())              # 'None' preserved as a real string

# Fix #2 (if you've already loaded the data): recover it explicitly, ONLY if you've
# confirmed via the data dictionary / domain knowledge that NaN here really does mean
# the category "None", not a genuine missing value.
demo2 = reloaded.copy()
demo2['damage'] = demo2['damage'].fillna('None')
print(demo2['damage'].tolist())

import os; os.remove('_demo.csv')


## 3. Unit-Bearing Text Cleaning

In [ ]:
# Comma-formatted thousands + unit suffix
pd.Series(['62,189 hrs', '9,489 hrs']).str.replace(',', '').str.replace(' hrs', '').astype(float)
pd.Series(['45,564 cycles']).str.replace(',', '').str.replace(' cycles', '').astype(float)
pd.Series(['2,637 kg/hr']).str.replace(',', '').str.replace(' kg/hr', '').astype(float)

# Extract a number embedded in a more complex string (regex)
pd.Series(['180 seats', 'Freighter (0 seats)']).str.extract(r'(\d+)').astype(int)

# Count + disguised-missing sentinel, same pattern as prior projects
raw = pd.Series(['2 ADs', 'Not Reported', '0 ADs'])
cleaned = raw.str.replace(' ADs', '').replace('Not Reported', '-1').astype(int)
unreported_flag = (cleaned == -1).astype(int)
imputed = cleaned.replace(-1, cleaned[cleaned != -1].median())


## 4. Ordinal Encoding for Naturally-Ordered Categories

In [ ]:
# Unlike one-hot encoding, ordinal encoding preserves a meaningful RANK.
# Use it when categories have a clear better/worse or more/less ordering.

check_map = {'D-Check Due': 0, 'C-Check Due': 1, 'Fresh C-Check': 2, 'Fresh D-Check': 3}
damage_map = {'Major': 0, 'Minor': 1, 'None': 2}
paint_map = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}

df_demo = pd.DataFrame({'check': ['C-Check Due', 'Fresh D-Check']})
df_demo['check_score'] = df_demo['check'].map(check_map)

# Rule of thumb: if you could sensibly ask "which of these two categories is BETTER/WORSE,
# not just DIFFERENT", it's a candidate for ordinal encoding instead of one-hot.


## 5. Domain-Derived Features

In [ ]:
# df['age_years'] = 2025 - df['manufacture_year']
# df['engine_overhaul_pct_remaining'] = 1 - (df['engine_hours_since_overhaul'] / 9000).clip(0, 1)
# df['is_freighter'] = (df['seat_count'] == 0).astype(int)

# A ratio feature normalizes a raw count by a relevant denominator, often more informative
# than either raw number alone:
# df['cycles_per_flight_hour'] = df['total_cycles'] / df['total_flight_hours']


## 6. EDA & the Market-Quote Leakage Check

In [ ]:
numeric_df = df.select_dtypes(include='number')
numeric_df.corr()['appraised_fair_market_value_usd'].sort_values(ascending=False)

df['appraised_fair_market_value_usd'].corr(df['broker_asking_price_usd'])          # expect very high
df['appraised_fair_market_value_usd'].corr(df['recent_comparable_sale_price_usd']) # expect even higher

# Log transform is often appropriate for asset VALUES specifically, because
# depreciation/appreciation is usually a PERCENTAGE process, not a flat-dollar one:
print("raw skew:", df['appraised_fair_market_value_usd'].skew())
print("log skew:", np.log(df['appraised_fair_market_value_usd']).skew())

from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = numeric_df.drop(columns=['appraised_fair_market_value_usd']).dropna()
vif = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)


## 7. Leakage-Safe Target Encoding

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
low_card = [c for c in cat_cols if df[c].nunique() < 5]      # e.g. maintenance_program, lease_status
high_card = [c for c in cat_cols if df[c].nunique() >= 5]    # e.g. aircraft_type, manufacturer, engine_type

df_enc = pd.get_dummies(df, columns=low_card, drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

from sklearn.model_selection import train_test_split
X = df_enc.drop(columns=['appraised_fair_market_value_usd'])   # + drop leakage-risk columns too
y = df_enc['appraised_fair_market_value_usd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)


## 8. Models, Including the Log-Linear Pattern

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

lin = LinearRegression().fit(X_train, y_train)

# Log-linear pattern: fit on log(target), then EXPONENTIATE predictions back to the
# original scale before computing any metric, so every model is compared fairly on
# the same (dollar) units.
lin_log = LinearRegression().fit(X_train, np.log(y_train))
pred_dollars = np.exp(lin_log.predict(X_test))
print("log-linear MAE ($):", mean_absolute_error(y_test, pred_dollars))
print("log-linear R2 ($ space):", r2_score(y_test, pred_dollars))

rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)


## 9. Cross-Validation & Tuning

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

scores = -cross_val_score(gb, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(scores.mean(), scores.std())

param_dist = {'n_estimators': [100, 200, 400], 'max_depth': [2, 3, 4, 5],
              'learning_rate': [0.01, 0.05, 0.1, 0.2]}
search = RandomizedSearchCV(GradientBoostingRegressor(random_state=RANDOM_STATE),
                             param_distributions=param_dist, n_iter=20, cv=5,
                             scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_


## 10. Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100   # safe here: values never near zero

plt.scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual value ($)'); plt.ylabel('Predicted value ($)')
plt.show()


## 11. Feature Importance

In [ ]:
importances = best_model.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]
sns.barplot(x=importances[top_idx], y=X_train.columns[top_idx])
plt.show()

from sklearn.inspection import permutation_importance
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_idx = result.importances_mean.argsort()[-10:][::-1]
sns.barplot(x=result.importances_mean[perm_idx], y=X_test.columns[perm_idx])
plt.show()


## 12. Deal-Finding Backtest Pattern

In [ ]:
test_df = df.loc[y_test.index].copy()
test_df['model_value'] = y_pred
test_df['edge'] = test_df['model_value'] - test_df['broker_asking_price_usd']

threshold = test_df['broker_asking_price_usd'] * 0.08   # flag if model sees >8% more value than asking
deals = test_df[test_df['edge'] > threshold]

# Confirmation check using a column that was NEVER a training feature
confirmed_rate = (deals['recent_comparable_sale_price_usd'] > deals['broker_asking_price_usd'] * 1.03).mean()
baseline_rate = (test_df['recent_comparable_sale_price_usd'] > test_df['broker_asking_price_usd'] * 1.03).mean()
print(f"Flagged {len(deals)}/{len(test_df)} | confirmed-undervalued rate {confirmed_rate:.1%} vs baseline {baseline_rate:.1%}")


## 13. Persistence & Inference

In [ ]:
import joblib

joblib.dump(best_model, 'aircraft_value_model.pkl')
joblib.dump({
    'target_encoding_maps': {}, 'model_columns': list(X_train.columns),
    'check_map': check_map, 'damage_map': damage_map, 'paint_map': paint_map,
}, 'aircraft_value_encoders.pkl')

def predict_fair_value(raw_aircraft_dict, model, encoders):
    # 1. put raw_aircraft_dict into a one-row DataFrame
    # 2. re-apply the SAME cleaning / ordinal maps / feature engineering / encoding used in training
    # 3. reindex to training column order, filling missing dummy columns with 0
    # 4. return model.predict(row)[0]
    pass


## Quick lookup: one-hot vs. ordinal vs. target encoding

| Situation | Encoding |
|---|---|
| Category has no natural order, few unique values (< 5) | One-hot |
| Category has no natural order, many unique values (>= 5) | Target/mean encoding (fit on train only) |
| Category has a clear better/worse ranking | Ordinal (hand-specified map) |
| A column is another format of the SAME thing you're predicting | Exclude entirely (leakage) |
